# Temporal action detection on THUMOS-14: Colab runner

**Before you start**
1. Upload `temporal-action-detection.zip` (the latest version) to the top level of **My Drive**.
2. Runtime → Change runtime type → **T4 GPU**.
3. Run the cells top to bottom. Every cell is safe to re-run, and anything already done (copied files, trained models) is skipped.

Everything runs on Colab's local disk (`/content`). Google Drive is only used to copy things in and back up results, because Drive connections drop during long copies.

## 1. Setup: mount Drive, unpack the code, define copy helpers

In [ ]:
import os, glob, shutil, time
os.chdir('/content')                      # never work inside Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE = '/content/drive/MyDrive'
FEAT_SRC = f'{DRIVE}/thumos_i3d/thumos_i3d/i3d_actionformer_stride4_thumos/i3d_actionformer_stride4_thumos'
CKPT_SRC = f'{DRIVE}/thumos14_checkpoints'    # the May (v1) models
BACKUP = f'{DRIVE}/tad_runs'                  # where results are backed up

def remount():
    os.chdir('/content')
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    time.sleep(5)
    drive.mount('/content/drive', force_remount=True)

def copy_file(src, dst, tries=5):
    """Copy one file, reconnecting Drive if it drops. Skips files already copied."""
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for _ in range(tries):
        try:
            if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
                return
            shutil.copyfile(src, dst)
            return
        except OSError:
            print(f'Drive dropped while copying {os.path.basename(src)}, reconnecting...')
            remount()
    raise RuntimeError(f'could not copy {src}')

def copy_dir(src, dst, pattern='*'):
    """Copy every file under src (recursively) to dst."""
    cwd = os.getcwd()
    os.chdir('/content')
    for _ in range(5):
        try:
            files = [p for p in glob.glob(os.path.join(src, '**', pattern), recursive=True) if os.path.isfile(p)]
            break
        except OSError:
            remount()
    for i, f in enumerate(sorted(files)):
        copy_file(f, os.path.join(dst, os.path.relpath(f, src)))
        if i and i % 100 == 0:
            print(f'  {i}/{len(files)}')
    os.chdir(cwd)
    return len(files)

# unpack the latest code into /content/tad (keeps any runs/ already there)
copy_file(f'{DRIVE}/temporal-action-detection.zip', '/content/repo.zip')
!rm -rf /content/new && unzip -qo /content/repo.zip -d /content/new
!mkdir -p /content/tad && cp -r /content/new/temporal-action-detection/. /content/tad/
os.chdir('/content/tad')
print('code ready in', os.getcwd())
!ls

## 2. Data: I3D features (from Drive) and THUMOS-14 annotations (from the official site)

In [ ]:
n = copy_dir(FEAT_SRC, '/content/features', '*.npy')
print('feature files on Drive:', n, '| copied locally:', len(os.listdir('/content/features')))

!mkdir -p /content/data/thumos14 && cd /content/data/thumos14 && \
  wget -q -nc --no-check-certificate https://www.crcv.ucf.edu/THUMOS14/Validation_set/TH14_Temporal_annotations_validation.zip && \
  wget -q -nc --no-check-certificate https://www.crcv.ucf.edu/THUMOS14/test_set/TH14_Temporal_annotations_test.zip && \
  unzip -qo -j TH14_Temporal_annotations_validation.zip -d annotations_val && \
  unzip -qo -j TH14_Temporal_annotations_test.zip -d annotations_test
print('annotation files:', len(os.listdir('/content/data/thumos14/annotations_val')), 'val,',
      len(os.listdir('/content/data/thumos14/annotations_test')), 'test')

## 3. Bring in the May checkpoints and any runs already trained (so nothing is retrained)

In [ ]:
print('v1 checkpoint files:', copy_dir(CKPT_SRC, '/content/ckpt'))
if os.path.exists(f'{BACKUP}/runs'):
    print('restored run files:', copy_dir(f'{BACKUP}/runs', '/content/tad/runs'))
done = sorted(d for d in os.listdir('runs') if os.path.exists(f'runs/{d}/results.json')) if os.path.exists('runs') else []
print('finished runs found:', done)

## 4. Before: re-score the May (v1) models with the standard evaluator
One line per model. A checkpoint that cannot be read is skipped (the May LSTM file is corrupted).

In [ ]:
!python scripts/rescore_old_models.py --ckpt_dir /content/ckpt

## 5. After: train the new detector (skipped if already trained)
About 20 min for the transformer neck and 13 min for the conv neck on a T4.

In [ ]:
def train(name, *overrides):
    if os.path.exists(f'runs/{name}/results.json'):
        print(f'{name}: already trained, skipping')
        return
    extra = ('--set ' + ' '.join(overrides)) if overrides else ''
    get_ipython().system(f'python train.py --config configs/thumos_i3d.yaml --out runs/{name} {extra}')

train('i3d_transformer')
train('i3d_conv', 'model.neck=conv')

## 6. Optional: two more seeds of the conv detector (about 25 min)
Lets you report mean ± standard deviation over 3 runs instead of a single number. Skip this cell if you are short on time.

In [ ]:
train('i3d_conv_seed1', 'model.neck=conv', 'train.seed=1')
train('i3d_conv_seed2', 'model.neck=conv', 'train.seed=2')

## 7. Summary over seeds

In [ ]:
import json, numpy as np

runs = [r for r in ['i3d_conv', 'i3d_conv_seed1', 'i3d_conv_seed2'] if os.path.exists(f'runs/{r}/results.json')]
print('runs found:', runs)

if not runs:
    print('No finished conv runs found. Check section 3 output (finished runs found: ...).')
else:
    res = [json.load(open(f'runs/{r}/results.json')) for r in runs]
    keys = ['0.3', '0.4', '0.5', '0.6', '0.7']
    table = {f'mAP@{k}': [r['mAP'][k] for r in res] for k in keys}
    table['avg_mAP'] = [r['avg_mAP'] for r in res]

    summary = {'runs': runs}
    print(f'\nconv detector over {len(runs)} run(s): mean ± std')
    for name, vals in table.items():
        v = np.array(vals, dtype=float)
        mean = float(v.mean())
        std = float(v.std(ddof=1)) if len(v) > 1 else 0.0
        summary[name] = {'mean': round(mean, 2), 'std': round(std, 2), 'per_run': vals}
        print(f'  {name:10s} {mean:5.1f} ± {std:.1f}   (runs: {", ".join(f"{x:.1f}" for x in vals)})')

    os.makedirs('results', exist_ok=True)
    json.dump(summary, open('results/seed_summary_conv.json', 'w'), indent=1)
    print('\nsaved results/seed_summary_conv.json')

## 8. Figures and the README results table

In [ ]:
!python scripts/make_figures.py --runs runs/i3d_conv runs/i3d_transformer --names "Detector, conv neck" "Detector, transformer neck" --baselines results/baselines.json
from IPython.display import Image, display
for f in ['figures/map_vs_tiou.png', 'figures/per_class_ap.png', 'figures/timeline_example.png']:
    display(Image(f))

## 9. Back up to Drive and download what goes into the GitHub repo
`for_repo.zip` contains `figures/`, `results/` and each run's `results.json` + `train_log.txt`. Unzip it into your local repo folder (model weights and raw predictions stay out of git).

In [ ]:
# backup (weights included) to Drive
for d in ['runs', 'results', 'figures']:
    for f in glob.glob(f'{d}/**/*', recursive=True):
        if os.path.isfile(f):
            copy_file(f, f'{BACKUP}/{f}')
print('backed up to', BACKUP)

# small bundle for the repo
!rm -f /content/for_repo.zip && zip -qr /content/for_repo.zip figures results runs/*/results.json runs/*/train_log.txt -x "runs/debug/*"
!unzip -l /content/for_repo.zip | tail -3
from google.colab import files
files.download('/content/for_repo.zip')

## Optional: smoke test (1 minute)
Only needed if you changed the code: trains 4 videos for 1 epoch to check that everything still runs.

In [ ]:
!python train.py --out runs/debug --max_videos 4 --set train.epochs=1 train.warmup_epochs=0